# TPC-H Lineitem Analysis

**Dataset:** `samples.tpch.lineitem`

**Difficulty:** Medium

**Topics:** calculated fields, aggregation, window, cumulative, complex filters

In [0]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

lineitem = spark.read.table("samples.tpch.lineitem")

## Problem 1

Calculate net revenue per line item using the formula:
`net_revenue = l_extendedprice * (1 - l_discount) * (1 + l_tax)`

**Expected output columns:**
- `l_orderkey`
- `l_linenumber`
- `l_extendedprice`
- `l_discount`
- `l_tax`
- `net_revenue`

In [0]:
lineitem.printSchema()

In [0]:
# Problem 1 - write your solution here
# Assign your result to: result_1
expr = "l_extendedprice * (1 - l_discount) * (1 + l_tax)"
result_1 = lineitem.select(
    "l_orderkey",
    "l_linenumber",
    "l_extendedprice",
    "l_discount",
    "l_tax",
    F.expr(expr).alias("net_revenue")
)
result_1.display()

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'l_orderkey' in cols, "Missing column: l_orderkey"
assert 'l_linenumber' in cols, "Missing column: l_linenumber"
assert 'l_extendedprice' in cols, "Missing column: l_extendedprice"
assert 'l_discount' in cols, "Missing column: l_discount"
assert 'l_tax' in cols, "Missing column: l_tax"
assert 'net_revenue' in cols, "Missing column: net_revenue"
assert len(cols) == 6, f"Expected exactly 6 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_rev = result_1.agg(F.min('net_revenue')).collect()[0][0]
assert min_rev >= 0, f"Expected net_revenue >= 0, found min={min_rev}"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

Compute total revenue by ship mode (`l_shipmode`), including average discount per mode.
Sort by `total_revenue` descending.

**Expected output columns:**
- `l_shipmode`
- `total_revenue`
- `avg_discount`

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2

result_2 = lineitem.withColumn(
    "revenue",
    F.expr(expr)
).groupBy("l_shipmode").agg(
    F.sum("revenue").alias("total_revenue"),
    F.avg("l_discount").alias("avg_discount")
).orderBy(F.col("total_revenue").desc())

result_2.display()

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'l_shipmode' in cols, "Missing column: l_shipmode"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'avg_discount' in cols, "Missing column: avg_discount"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_rev = result_2.agg(F.min('total_revenue')).collect()[0][0]
assert min_rev >= 0, f"Expected total_revenue >= 0, found min={min_rev}"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Count line items by return flag (`l_returnflag`) and line status (`l_linestatus`), also computing total quantity and total revenue.

**Expected output columns:**
- `l_returnflag`
- `l_linestatus`
- `line_count`
- `total_qty`
- `total_revenue`

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3

result_3 = lineitem.withColumn(
    "revenue",
    F.expr(expr)
).groupBy("l_returnflag", "l_linestatus").agg(
    F.count("*").alias("line_count"),
    F.sum("l_quantity").alias("total_qty"),
    F.sum("revenue").alias("total_revenue")
)

result_3.display()

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'l_returnflag' in cols, "Missing column: l_returnflag"
assert 'l_linestatus' in cols, "Missing column: l_linestatus"
assert 'line_count' in cols, "Missing column: line_count"
assert 'total_qty' in cols, "Missing column: total_qty"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_line = result_3.agg(F.min('line_count')).collect()[0][0]
assert min_line > 0, f"Expected line_count > 0, found min={min_line}"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

Aggregate by ship date month: compute total revenue, total quantity, and average discount.
Extract year and month from `l_shipdate`.

**Expected output columns:**
- `ship_year`
- `ship_month`
- `total_revenue`
- `total_quantity`
- `avg_discount`

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4

result_4 = lineitem.withColumn(
    "revenue",
    F.expr(expr)
).groupBy(
    F.year("l_shipdate").alias("ship_year"),
    F.month("l_shipdate").alias("ship_month")
).agg(
    F.sum("revenue").alias("total_revenue"),
    F.sum("l_quantity").alias("total_quantity"),
    F.avg("l_discount").alias("avg_discount")
)

result_4.display()

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'ship_year' in cols, "Missing column: ship_year"
assert 'ship_month' in cols, "Missing column: ship_month"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'total_quantity' in cols, "Missing column: total_quantity"
assert 'avg_discount' in cols, "Missing column: avg_discount"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_month = result_4.agg(F.max('ship_month')).collect()[0][0]
assert max_month <= 12, f"Expected ship_month <= 12, got max={max_month}"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

Find items where discount > 0.05 and quantity > 20 (potential bulk discounts).

**Expected output columns:**
- `l_orderkey`
- `l_partkey`
- `l_quantity`
- `l_discount`
- `l_extendedprice`

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5

result_5 = lineitem.filter(
    (F.col("l_discount") > 0.05) &
    (F.col("l_quantity") > 20)
).select(
    "l_orderkey",
    "l_partkey",
    "l_quantity",
    "l_discount",
    "l_extendedprice"
)

result_5.limit(5).display()

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'l_orderkey' in cols, "Missing column: l_orderkey"
assert 'l_partkey' in cols, "Missing column: l_partkey"
assert 'l_quantity' in cols, "Missing column: l_quantity"
assert 'l_discount' in cols, "Missing column: l_discount"
assert 'l_extendedprice' in cols, "Missing column: l_extendedprice"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_disc = result_5.agg(F.min('l_discount')).collect()[0][0]
assert float(min_disc) > 0.05, f"Expected l_discount > 0.05, found min={min_disc}"
min_qty = result_5.agg(F.min('l_quantity')).collect()[0][0]
assert float(min_qty) > 20, f"Expected l_quantity > 20, found min={min_qty}"
print(f"Problem 5 passed ✓  ({cnt} rows)")

## Problem 6

Using a window function, compute the cumulative revenue ordered by `l_shipdate` within each ship mode.

**Expected output columns:**
- `l_shipdate`
- `l_shipmode`
- `l_extendedprice`
- `cumulative_revenue`

In [0]:
# Problem 6 - write your solution here
# Assign your result to: result_6
w = Window.partitionBy("l_shipmode")
result_6 = lineitem.select(
    "l_shipdate",
    "l_shipmode",
    "l_extendedprice",
    F.sum(F.expr(expr)).over(w).alias("cumulative_revenue")
).orderBy("cumulative_revenue")

result_6.limit(10).display()

In [0]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None - did you assign your DataFrame?"
assert hasattr(result_6, 'columns'), "result_6 must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'l_shipdate' in cols, "Missing column: l_shipdate"
assert 'l_shipmode' in cols, "Missing column: l_shipmode"
assert 'l_extendedprice' in cols, "Missing column: l_extendedprice"
assert 'cumulative_revenue' in cols, "Missing column: cumulative_revenue"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_cum = result_6.agg(F.min('cumulative_revenue')).collect()[0][0]
assert float(min_cum) >= 0, f"Expected cumulative_revenue >= 0, found min={min_cum}"
print(f"Problem 6 passed ✓  ({cnt} rows)")

## Problem 7

Find the average number of line items per order, then identify orders that have more line items than the average.

**Expected output columns:**
- `l_orderkey`
- `line_item_count`
- `avg_line_items`

In [0]:
# Problem 7 - write your solution here
# Assign your result to: result_7
w = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
result_7 = lineitem.groupBy("l_orderkey").agg(
    F.count("*").alias("line_item_count")
).withColumn(
    "avg_line_items",
    F.avg("line_item_count").over(w).alias("avg_line_items")
).filter(
    F.col("line_item_count") > F.col("avg_line_items")
)

result_7.limit(10).display()

In [0]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None - did you assign your DataFrame?"
assert hasattr(result_7, 'columns'), "result_7 must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'l_orderkey' in cols, "Missing column: l_orderkey"
assert 'line_item_count' in cols, "Missing column: line_item_count"
assert 'avg_line_items' in cols, "Missing column: avg_line_items"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
invalid = result_7.filter(F.col('line_item_count') <= F.col('avg_line_items')).count()
assert invalid == 0, f"Found {invalid} rows where line_item_count <= avg_line_items"
print(f"Problem 7 passed ✓  ({cnt} rows)")